## Part 1: Preprocessing

In [1]:
# Import our dependencies
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
import pandas as pd
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras import layers

#  Import and read the attrition data
attrition_df = pd.read_csv('https://static.bc-edx.com/ai/ail-v-1-0/m19/lms/datasets/attrition.csv')
attrition_df.head()

,Age,Attrition,BusinessTravel,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,HourlyRate,JobInvolvement,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,Sales,1,2,Life Sciences,2,94,3,...,3,1,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,Research & Development,8,1,Life Sciences,3,61,2,...,4,4,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,Research & Development,2,2,Other,4,92,2,...,3,2,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,Research & Development,3,4,Life Sciences,4,56,3,...,3,3,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,Research & Development,2,1,Medical,1,40,3,...,3,4,1,6,3,3,2,2,2,2


In [2]:
# Determine the number of unique values in each column.
attrition_df.nunique()

Age                         43
Attrition                    2
BusinessTravel               3
Department                   3
DistanceFromHome            29
Education                    5
EducationField               6
EnvironmentSatisfaction      4
HourlyRate                  71
JobInvolvement               4
JobLevel                     5
JobRole                      9
JobSatisfaction              4
MaritalStatus                3
NumCompaniesWorked          10
OverTime                     2
PercentSalaryHike           15
PerformanceRating            2
RelationshipSatisfaction     4
StockOptionLevel             4
TotalWorkingYears           40
TrainingTimesLastYear        7
WorkLifeBalance              4
YearsAtCompany              37
YearsInCurrentRole          19
YearsSinceLastPromotion     16
YearsWithCurrManager        18
dtype: int64

In [3]:
# Create y_df with the Attrition and Department columns
y_df = attrition_df[['Attrition', 'Department']]
y_df.head()


,Attrition,Department
0,Yes,Sales
1,No,Research & Development
2,Yes,Research & Development
3,No,Research & Development
4,No,Research & Development


In [4]:
# Create a list of at least 10 column names to use as X data
columns_list = ['BusinessTravel', 'DistanceFromHome', 'Education', 'EducationField', 'EnvironmentSatisfaction', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobRole', 'JobSatisfaction']


# Create X_df using your selected columns
X_df = pd.DataFrame(attrition_df[columns_list])

# Show the data types for X_df
X_df.dtypes


BusinessTravel             object
DistanceFromHome            int64
Education                   int64
EducationField             object
EnvironmentSatisfaction     int64
HourlyRate                  int64
JobInvolvement              int64
JobLevel                    int64
JobRole                    object
JobSatisfaction             int64
dtype: object

In [5]:
# Split the data into training and testing sets
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_df, y_df)


In [12]:
# Convert your X data to numeric data types however you see fit
# Add new code cells as necessary
ordinal = OrdinalEncoder()

X_train['BusinessTravel'] = ordinal.fit_transform(X_train[['BusinessTravel']])
X_test['BusinessTravel'] = ordinal.fit_transform(X_test[['BusinessTravel']])

X_train['EducationField'] = ordinal.fit_transform(X_train[['EducationField']])
X_test['EducationField'] = ordinal.fit_transform(X_test[['EducationField']])

X_train['JobRole'] = ordinal.fit_transform(X_train[['JobRole']])
X_test['JobRole'] = ordinal.fit_transform(X_test[['JobRole']])


In [13]:
X_train.dtypes

BusinessTravel             float64
DistanceFromHome             int64
Education                    int64
EducationField             float64
EnvironmentSatisfaction      int64
HourlyRate                   int64
JobInvolvement               int64
JobLevel                     int64
JobRole                    float64
JobSatisfaction              int64
dtype: object

In [14]:
# Create a StandardScaler
scaler = StandardScaler()

# Fit the StandardScaler to the training data
scaler.fit_transform(X_train)

# Scale the training and testing data

X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [16]:
# Create a OneHotEncoder for the Department
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output = False)
# Create a OneHotEncoder for the Department column
# Fit the encoder to the training data
# Create two new variables by applying the encoder
# to the training and testing data
y_depart_train = y_train['Department']
y_depart_test = y_test['Department']

#Encode the Department

y_depart_train  = encoder.fit_transform(np.array(y_depart_train).reshape(-1,1))
y_depart_test = encoder.fit_transform(np.array(y_depart_test).reshape(-1,1))

y_depart_train

array([[0., 0., 1.],
       [0., 1., 0.],
       [0., 0., 1.],
       ...,
       [0., 1., 0.],
       [0., 1., 0.],
       [0., 1., 0.]])

In [18]:
# Create a OneHotEncoder for the Attrition column
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output = False)

attrition_y_train = y_train['Attrition']
attrition_y_test = y_test['Attrition']
# Fit the encoder to the training data
attrition_y_train = encoder.fit_transform(np.array(y_train[['Attrition']]))

# Transform the testing data
attrition_y_test = encoder.fit_transform(np.array(y_test[['Attrition']]))
#attrition_y_test
attrition_y_train


array([[1., 0.],
       [1., 0.],
       [0., 1.],
       ...,
       [1., 0.],
       [0., 1.],
       [1., 0.]])

## Create, Compile, and Train the Model

In [20]:
# Find the number of columns in the X training data
x_train_columns = len(X_train.columns)
# 10

# Create the input layer
input_layer = layers.Input(shape=(len(X_train.columns),), name='input_features')

# Create at least two shared layers
shared_layer1 = layers.Dense(64, activation='relu')(input_layer)
shared_layer2 = layers.Dense(128, activation='relu')(shared_layer1)

In [21]:
# Create a branch for Department
# with a hidden layer and an output layer
# Create the hidden layer
department_hidden1 = layers.Dense(32, activation='softmax')(shared_layer2)

# Create the output layer
department_output = layers.Dense(3, activation='softmax', name='department_output')(department_hidden1)


In [23]:
# Create a branch for Attrition
# with a hidden layer and an output layer
# num_attrition_classes = attrition_y_test.unique()
# Create the hidden layer
attrition_hidden1 = layers.Dense(32, activation='softmax')(shared_layer2)

# Create the output layer
attrition_output = layers.Dense(2, activation='softmax', name='attrition_output')(attrition_hidden1)



In [24]:
# Create the model
model = Model(inputs=input_layer, outputs=[department_output, attrition_output])

# Compile the model
model.compile(optimizer='adam',
              loss={'department_output': 'categorical_crossentropy', 'attrition_output': 'binary_crossentropy'},
              metrics={'department_output': 'accuracy', 'attrition_output': 'accuracy'})

# Summarize the model
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_features      │ (None, 10)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │        704 │ input_features[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │      8,320 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 32)        │      4,128 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 32)        │      4,128 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ department_output   │ (None, 3)         │         99 │ dense_2[0][0]     │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attrition_output    │ (None, 2)         │         66 │ dense_3[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 17,445 (68.14 KB)

 Trainable params: 17,445 (68.14 KB)

 Non-trainable params: 0 (0.00 B)

In [25]:
# Train the model
model.fit(
    X_train_scaled,
    {'department_output': y_depart_train, 'attrition_output': attrition_y_train},
    epochs=10,
    batch_size=32,
    validation_split=0.2
)


Epoch 1/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - attrition_output_accuracy: 0.8533 - department_output_accuracy: 0.4049 - loss: 1.7611 - val_attrition_output_accuracy: 0.8371 - val_department_output_accuracy: 0.6923 - val_loss: 1.6293
Epoch 2/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - attrition_output_accuracy: 0.8591 - department_output_accuracy: 0.6477 - loss: 1.5907 - val_attrition_output_accuracy: 0.8371 - val_department_output_accuracy: 0.6923 - val_loss: 1.4386
Epoch 3/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - attrition_output_accuracy: 0.8506 - department_output_accuracy: 0.6402 - loss: 1.4527 - val_attrition_output_accuracy: 0.8371 - val_department_output_accuracy: 0.6923 - val_loss: 1.3846
Epoch 4/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - attrition_output_accuracy: 0.8616 - department_output_accuracy: 0.6523 - loss: 1.3988 - val_attrition_output_accuracy: 0.8371 - val_department_output_accuracy: 0.6923 - val_loss: 1.3531
Epoch 5/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/st

In [26]:
# Evaluate the model with the testing data
test_results = model.evaluate(X_test, {'department_output': y_depart_test, 'attrition_output': attrition_y_test})
test_results

12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - attrition_output_accuracy: 0.8284 - department_output_accuracy: 0.6477 - loss: 1.3073 


[1.297395944595337, 0.8070651888847351, 0.6875]

In [31]:
# Print the accuracy for both department and attrition
test_results = model.evaluate(X_test, {'attrition_output' : attrition_y_test, 'department_output' : y_depart_test})
print(f"Attrition predictions accuracy: {test_results[1]}")
print(f"Department predictions accuracy: {test_results[2]}")


12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - attrition_output_accuracy: 0.8284 - department_output_accuracy: 0.6477 - loss: 1.3073 
Attrition predictions accuracy: 0.8070651888847351
Department predictions accuracy: 0.6875


# Summary

In the provided space below, briefly answer the following questions.

1. Is accuracy the best metric to use on this data? Why or why not?
accuracy is not that helpful for catagorical data. also the dataset isnt balenced

2. What activation functions did you choose for your output layers, and why?
I tried relu, sigmoid, and softmax before ultimately going with softmax to get the code to compile. softmax offered multi-classing for the dataset. sigmoid and relu never compiled and kept giving me shape errors.

3. Can you name a few ways that this model might be improved?
running the comparisons separately might have produced higher accuracy in the Department column, the nearly 50% accuracy we are currently seeing seems to indicate there is a lot of noise in the columns. a correlation matrix being run before the 10 columns were selected would have identified which 10 columns would have provided the best accuracy.

YOUR ANSWERS HERE

1. No, accuracy is not that helpful for catagorical data. also the dataset isnt balenced for the attrition and department columns. balenced accuracy would be better than accuracy in this dataset.
2. I tried relu, sigmoid, and softmax before ultimately going with softmax to get the code to compile. softmax offered multi-classing for the 3 different departments. sigmoid and relu never compiled probably due to needing softmax's multiclassing
3. running the comparisons separately might have produced higher accuracy in the Department column, the nearly 50% accuracy we are currently seeing seems to indicate there is a lot of noise in the columns. a correlation matrix being run before the 10 columns were selected would have identified which 10 columns would have provided the best accuracy. 